# CNN Picture Classification

## Обзор
Этот ноутбук демонстрирует полный цикл обучения модели для классификации изображений.

In [ ]:
# Импорт библиотек
import sys
sys.path.append('..')

import torch
import matplotlib.pyplot as plt
from pathlib import Path

from configs.config import Config
from src.data_loader import get_data_loaders, analyze_dataset
from src.model import create_model
from src.train import train_model
from src.utils import plot_training_history, plot_metrics, evaluate_model

## 1. Конфигурация

In [ ]:
config = Config()
print(config)

## 2. Анализ датасета

In [ ]:
dataset_stats = analyze_dataset(config.TRAIN_DIR)
print(f"Total images: {dataset_stats['total_images']}")
print(f"Number of classes: {dataset_stats['num_classes']}")
print(f"Class distribution: {dataset_stats['class_counts']}")

## 3. Загрузка данных

In [ ]:
train_loader, test_loader, class_names = get_data_loaders(
    train_dir=config.TRAIN_DIR,
    test_dir=config.TEST_DIR,
    batch_size=config.BATCH_SIZE,
    image_size=config.IMAGE_SIZE,
    num_workers=config.NUM_WORKERS
)
print(f"Classes: {class_names}")

## 4. Создание модели

In [ ]:
model = create_model(num_classes=config.NUM_CLASSES)
print(model)

## 5. Обучение

In [ ]:
save_path = config.OUTPUT_DIR / 'best_model.pth'
history = train_model(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    config=config,
    save_path=save_path
)

## 6. Визуализация результатов

In [ ]:
plot_training_history(
    history,
    save_path=config.OUTPUT_DIR / 'training_history.png'
)

In [ ]:
# Загрузка лучшей модели
model.load_state_dict(torch.load(save_path))

# Оценка
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
y_true, y_pred, y_probs = evaluate_model(model, test_loader, device)

# Визуализация метрик
plot_metrics(
    y_true, y_pred, class_names,
    save_path=config.OUTPUT_DIR / 'classification_metrics.png'
)